In [60]:
import requests
import pandas as pd

adm4 = "31.71.03.1001"

start_time = "2026-04-27 00:00:00"
end_time   = "2026-04-29 23:59:59"
path=r"D:\Data Pipeline\Web Scrapping\Output_BMKG"
url = f"https://api.bmkg.go.id/publik/prakiraan-cuaca?adm4={adm4}"

response = requests.get(url)
response.raise_for_status()
data = response.json()

all_times = []

for hari in data["data"][0]["cuaca"]:
    for jam in hari:
        all_times.append(pd.to_datetime(jam.get("local_datetime")))

min_time = min(all_times)
max_time = max(all_times)

print(f"Range data tersedia: {min_time} sampai {max_time}")

rows = []

start = pd.to_datetime(start_time)
end   = pd.to_datetime(end_time)

for i, hari in enumerate(data["data"][0]["cuaca"]):
    for jam in hari:
        jam_datetime = pd.to_datetime(jam.get("local_datetime"))
        
        if start <= jam_datetime <= end:
            rows.append({
                "hari_ke": i + 1,
                "waktu": jam_datetime,
                "suhu_C": jam.get("t"),
                "kelembapan_%": jam.get("hu"),
                "kecepatan_angin_km_per_jam": jam.get("ws"),
                "arah_angin": jam.get("wd"),
                "tutupan_awan_%": jam.get("tcc"),
                "visibility": jam.get("vs_text"),
                "cuaca": jam.get("weather_desc")
            })

df = pd.DataFrame(rows)

if df.empty:
    print("WARNING: Data kosong! Range waktu tidak sesuai dengan data BMKG.")
else:
    print(f"Total data terambil: {len(df)}")

Range data tersedia: 2026-04-27 11:00:00 sampai 2026-04-29 23:00:00
Total data terambil: 21


In [61]:
lokasi = data["lokasi"]
kecamatan = lokasi.get("kecamatan")
file_name = fr"{path}\data_cuaca_bmkg_{kecamatan}.xlsx"
df.to_excel(file_name, index=False)

print(f"File berhasil disimpan: {file_name}")

File berhasil disimpan: D:\Data Pipeline\Web Scrapping\Output_BMKG\data_cuaca_bmkg_Kemayoran.xlsx


In [62]:
with pd.ExcelWriter(fr"{path}\data_cuaca_bmkg_{kecamatan}.xlsx", engine="openpyxl") as writer:

    df.to_excel(writer, sheet_name="Cuaca", index=False)
    
    df_lokasi = pd.DataFrame([{
        "Desa": lokasi.get("desa"),
        "Kecamatan": lokasi.get("kecamatan"),
        "Kota": lokasi.get("kotkab"),
        "Provinsi": lokasi.get("provinsi"),
        "Latitude": lokasi.get("lat"),
        "Longitude": lokasi.get("lon"),
        "Timezone": lokasi.get("timezone")
    }])
    
    df_lokasi.to_excel(writer, sheet_name="Lokasi", index=False)

print("Excel lengkap (multi-sheet) berhasil dibuat!")

Excel lengkap (multi-sheet) berhasil dibuat!


In [63]:
rows = []

for hari in data["data"][0]["cuaca"]:
    for jam in hari:
        rows.append({
            "waktu": jam.get("local_datetime"),
            "cuaca": jam.get("weather_desc"),
            "suhu": jam.get("t"),
            "kelembapan": jam.get("hu"),
            "angin": jam.get("ws"),
            "arah_angin": jam.get("wd"),
            "visibility": jam.get("vs_text")
        })

df = pd.DataFrame(rows)

df.head()

,waktu,cuaca,suhu,kelembapan,angin,arah_angin,visibility
0,2026-04-27 11:00:00,Berawan,32,64,9.4,N,> 10 km
1,2026-04-27 14:00:00,Berawan,30,69,2.7,NW,> 10 km
2,2026-04-27 17:00:00,Berawan,30,76,2.8,NE,> 10 km
3,2026-04-27 20:00:00,Berawan,28,83,2.8,NE,> 10 km
4,2026-04-27 23:00:00,Berawan,27,88,1.4,SW,> 10 km
